# Waveform Propagation Examples

This notebook demonstrates the new architecture for scalar field waveform propagation through galactic density profiles.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from constants import SEC_TO_INEV
from utils.data_utils import read_medium_data, interpolate_data
from inputs.spectrum_sources import (CSVSpectrum, AnalyticSpectrum, SpectrumSource,
                                      TimeVaryingSpectrum, CompositeSpectrum, plot_spectrum)
from inputs.configs import PhysicsConfig, PropagationConfig
from utils.logging_utils import setup_logging, get_logger
from waveform.collection import WaveformCollection
from waveform.propagation import plot_spectrogram

# Enable inline plotting
%matplotlib inline

# Configure logging
setup_logging(log_file='propagation.log', level='INFO')
logger = get_logger()

## Example 1: Single Static Gaussian

Demonstrates a single Gaussian wavepacket propagating through the galactic density profile.

In [ ]:
mass = 1e-19
burst_duration = 2 * np.pi * 100 / (mass * SEC_TO_INEV)

# Create spectrum source
spectrum = AnalyticSpectrum(
    'gaussian',
    mass=mass,
    num_points=1000,
    peak_momentum=2e2 * mass,
    width=2e1 * mass,
    amplitude=1e50
)

# Optionally plot the initial spectrum before propagation
plot_spectrum(spectrum, filename='gaussian_initial.pdf')
print("✓ Initial spectrum saved: gaussian_initial.pdf")

# Configure physics and propagation
physics = PhysicsConfig(mass=mass, coupling=1e20, K=1e-3, burst_duration=burst_duration)
propagation_config = PropagationConfig('Galactic_Density_Profile.csv', density_num_points=2000)

# Create collection and propagate
collection = WaveformCollection(spectrum, physics, propagation_config)
results = collection.propagate_all(N_points_spectrogram=2000)

print("✓ Waveform saved: waveform_plot.pdf")

# Plot spectrogram
plot_spectrogram(results['N_points'], results['t_min'], results['t_max'],
                 results['E'], results['spectrogram'], results['valid'])
print("✓ Spectrogram saved: spectrogram_plot.pdf")

logger.info("Single Gaussian propagation complete. Spectrogram saved.")

## Example 2: Time-Varying Gaussian

Demonstrates a Gaussian wavepacket with time-varying amplitude modulation.

In [ ]:
mass = 1e-19
burst_duration = 2 * np.pi * 100 / (mass * SEC_TO_INEV)

# Create base Gaussian spectrum
base_gaussian = AnalyticSpectrum(
    'gaussian',
    mass=mass,
    num_points=1000,
    peak_momentum=2e2 * mass,
    width=2e1 * mass,
    amplitude=1e50
)

# Create time-varying wrapper with Gaussian amplitude profile
N_time_steps = 20
amplitude_profile = [np.exp(-(i - N_time_steps/2)**2 / (N_time_steps/3)**2)
                     for i in range(N_time_steps)]

# Define time windows for each step
time_windows = [(i * burst_duration * SEC_TO_INEV, (i + 1) * burst_duration * SEC_TO_INEV)
                for i in range(N_time_steps)]

time_varying = TimeVaryingSpectrum(base_gaussian, amplitude_profile, time_windows)

# Configure and propagate
physics = PhysicsConfig(mass=mass, coupling=1e20, K=1e-3, burst_duration=burst_duration)
propagation_config = PropagationConfig('Galactic_Density_Profile.csv', density_num_points=2000)

collection = WaveformCollection(time_varying, physics, propagation_config)
results = collection.propagate_all(N_points_spectrogram=2000, save_waveform=False)

# Plot spectrogram
plot_spectrogram(results['N_points'], results['t_min'], results['t_max'],
                 results['E'], results['spectrogram'], results['valid'])
print("✓ Spectrogram saved: spectrogram_plot.pdf")

logger.info(f"Time-varying propagation complete with {N_time_steps} steps")

## Example 3: Composite Spectrum (Multiple Emission Lines)

Demonstrates multiple independent Gaussian emission lines that are propagated separately and combined in the spectrogram.

In [ ]:
mass = 1e-19
burst_duration = 2 * np.pi * 100 / (mass * SEC_TO_INEV)

# Create two Gaussian emission lines at different energies
# Both will be propagated independently and combined in the spectrogram
gaussian1 = AnalyticSpectrum(
    'gaussian',
    mass=mass,
    num_points=1000,
    peak_momentum=1e2 * mass,
    width=1e1 * mass,
    amplitude=4e50
)

gaussian2 = AnalyticSpectrum(
    'gaussian',
    mass=mass,
    num_points=1000,
    peak_momentum=2e2 * mass,
    width=1e1 * mass,
    amplitude=5e50
)

# Combine into composite spectrum
composite = CompositeSpectrum([gaussian1, gaussian2])

# Optionally plot the initial composite spectrum before propagation
plot_spectrum(composite, filename='composite_initial.pdf', xlabel='Momentum (eV)', ylabel='Amplitude')
print("✓ Initial composite spectrum saved: composite_initial.pdf")

# Configure and propagate
physics = PhysicsConfig(mass=mass, coupling=1e20, K=1e-3, burst_duration=burst_duration)
propagation_config = PropagationConfig('Galactic_Density_Profile.csv', density_num_points=2000)

collection = WaveformCollection(composite, physics, propagation_config)
results = collection.propagate_all(N_points_spectrogram=2000, save_waveform=True)

print("✓ Composite waveform saved (both sources overlaid): waveform_plot.pdf")

# Plot spectrogram
plot_spectrogram(results['N_points'], results['t_min'], results['t_max'],
                 results['E'], results['spectrogram'], results['valid'])
print("✓ Spectrogram saved: spectrogram_plot.pdf")

logger.info("Composite spectrum propagation complete. Spectrogram saved.")

## Example 4: CSV Spectrum (Bosenova)

Demonstrates loading a spectrum from a CSV file (boson star collapse spectrum).

In [ ]:
mass = 1e-19
burst_duration = 400 / (mass * SEC_TO_INEV)

# Load bosenova spectrum from file (no header, space-separated)
# Use num_points=3000 for interpolation to match old implementation
spectrum = CSVSpectrum('Spectra/BosonStarSpectrumRelOnly.txt',
                       i_p=0, i_A=1, skip_header=False, num_points=3000)

# The file contains dimensionless values - need to scale properly
# Following old code normalization: momenta * mass, amplitudes * sqrt(1/1e-85)
momenta, amplitudes = spectrum.get_spectrum()
scaled_momenta = momenta * mass
scaled_amplitudes = np.sqrt(amplitudes * (1/1e-85))  # Match old BosenovaSpectrum normalization

# Create a wrapper class to return scaled values
class ScaledSpectrum(SpectrumSource):
    def get_spectrum(self, time_index=None):
        return scaled_momenta, scaled_amplitudes

scaled_spectrum = ScaledSpectrum()

physics = PhysicsConfig(mass=mass, coupling=1.5e22, K=1e-3, burst_duration=burst_duration)
propagation_config = PropagationConfig('Galactic_Density_Profile.csv', density_num_points=1000)

# For bosenova, need to scale the density profile distance from 10kpc to 1pc (x/10000) before creating collection
# We'll create a custom WaveformCollection with modified density loading
collection = WaveformCollection(scaled_spectrum, physics, propagation_config)
# Override the density profile to apply bosenova scaling
x, rho = read_medium_data('Galactic_Density_Profile.csv', i_R=0, i_rho=2)
x_interp, rho_interp = interpolate_data(x/10000, rho, 1000)  # Bosenova: Convert 10 kpc to 1 pc
collection.density_profile = [x_interp, rho_interp]

results = collection.propagate_all(N_points_spectrogram=2000, save_waveform=False)

# Plot spectrogram
plot_spectrogram(results['N_points'], results['t_min'], results['t_max'],
                 results['E'], results['spectrogram'], results['valid'])
print("✓ Spectrogram saved: spectrogram_plot.pdf")

logger.info("CSV (Bosenova) spectrum propagation complete. Spectrogram saved.")

## Summary

This notebook demonstrates the four main use cases of the new waveform propagation architecture:

1. **Single static spectrum**: Basic Gaussian wavepacket
2. **Time-varying spectrum**: Amplitude modulation over time
3. **Composite spectrum**: Multiple independent emission lines
4. **CSV spectrum**: Loading external data (bosenova)

Each example produces:
- Initial spectrum plot (optional)
- Waveform plot showing φ(t) at Earth
- Spectrogram showing frequency vs. time evolution